# M1 Lab — Orientation & Intro to Data Science

**Dataset:** `sales_monthly.csv` &nbsp;|&nbsp; **Anchors:** McKinney Ch1, Géron Ch1

No `[AI-OFF]` cells in M1. Use this lab to set up your environment and practice the disclosure workflow.

In [ ]:
MODULE_ID = "M1"
# ── Confusion reporter ───────────────────────────────────────────────────────
# Run this cell once to set up the reporter, then call it any time you are
# confused about a term or concept. It logs the entry to the instructor
# dashboard so they can address common pain-points.
#
#   Usage (in any later cell):
#       await im_confused("overfitting")
#       await im_confused("gradient descent", "not sure how the learning rate affects convergence")

import json as _json

async def im_confused(term: str, note: str = ""):
    """Report a confusing term to your instructor.
    
    Args:
        term: the word / concept that confused you (e.g. "overfitting")
        note: optional extra detail (e.g. "what does the bias-variance tradeoff mean here?")
    """
    try:
        from pyodide.http import pyfetch
        payload = {
            "word": term,
            "message": note,
            "source": "self_report",
            "moduleId": MODULE_ID,
        }
        resp = await pyfetch(
            "/api/error-log",
            method="POST",
            headers={"Content-Type": "application/json"},
            body=_json.dumps(payload),
            credentials="include",
        )
        if resp.ok:
            print(f"✅ Reported \"{term}\" — your instructor will see this in the Confusion dashboard.")
        else:
            print(f"⚠️  Could not report (HTTP {resp.status}). Are you logged in to DataPath?")
    except ImportError:
        # Running outside JupyterLite (e.g. plain Jupyter / Docker)
        import requests as _req
        payload = {"word": term, "message": note, "source": "self_report", "moduleId": MODULE_ID}
        try:
            r = _req.post("http://localhost:3001/api/error-log", json=payload, timeout=5)
            print("✅ Reported!" if r.ok else f"⚠️  HTTP {r.status_code}")
        except Exception as e:
            print(f"⚠️  Could not reach DataPath server: {e}")
    except Exception as e:
        print(f"⚠️  Unexpected error: {e}")

print("✓ Confusion reporter ready.")
print("  Usage: await im_confused(\"term you found confusing\")")


## L1.4 — Reproducibility setup `[Expert]`

In [ ]:
import numpy as np, random, pandas as pd
np.random.seed(42)
random.seed(42)

# Notebook header — copy to every future notebook
# Author: [your name]
# Dataset: sales_monthly.csv
# Environment: ds-gemma-course

## L1.3 — Load and inspect `[Expert]`

In [ ]:
df = pd.read_csv('sales_monthly.csv')
print(df.shape)
df.head()

## L1.3 — Pipeline questions (write in markdown)

Answer in the markdown cell below:
1. What question does this dataset help us answer?
2. What cleaning steps might be needed (guess before looking)?
3. What would a useful insight look like?

*(your answers here)*

## L1.5 — Correlation vs causation discussion `[Expert]`

Pick any two columns that look related. In a markdown cell, propose:
- The correlation you would expect
- A plausible **third** variable that could be the actual cause

*(your hypothesis here)*

---
## Submission checklist
- [ ] All cells run from a fresh kernel
- [ ] `AI_USE.md` initialised (even if empty for M1)
- [ ] Reproducibility block present in cell 1

## L1.7 — Local AI Workspace Setup


### Step 1: Ollama health check

Run this cell first. It verifies that Ollama is running. If you get a connection error, make sure Docker Compose is running (or Ollama is started on your machine).


In [ ]:
# Works in JupyterLite (pyfetch) AND Docker JupyterLab (requests)
import json as _json

async def check_ollama():
    """Check if Ollama is running and list available models."""
    try:
        from pyodide.http import pyfetch          # JupyterLite path
        resp = await pyfetch("http://localhost:11434/api/tags")
        data = await resp.json()
    except ImportError:
        import requests                            # Docker / server Jupyter path
        data = requests.get("http://localhost:11434/api/tags", timeout=5).json()

    models = data.get("models", [])
    if models:
        print("✅ Ollama is running!")
        print("Available models:")
        for m in models:
            print(f"  - {m['name']}")
    else:
        print("⚠️  Ollama is running but no models pulled yet.")
        print("   Run: ollama pull gemma4:e2b")

await check_ollama()


### Step 2: First chat completion (non-streaming)

This sends a prompt to Gemma 4n and waits for the full response. Simpler to debug than streaming.


In [ ]:
import json as _json

async def ask_gemma(prompt: str, model: str = "gemma4:e2b") -> str:
    """Send a prompt to Ollama. Works in JupyterLite (pyfetch) and Docker (requests)."""
    try:
        from pyodide.http import pyfetch          # JupyterLite path
        resp = await pyfetch(
            "http://localhost:11434/api/chat",
            method="POST",
            headers={"Content-Type": "application/json"},
            body=_json.dumps({
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "stream": False,
            }),
        )
        data = await resp.json()
    except ImportError:
        import requests                            # Docker / server Jupyter path
        r = requests.post(
            "http://localhost:11434/api/chat",
            json={"model": model, "messages": [{"role": "user", "content": prompt}], "stream": False},
            timeout=120,
        )
        r.raise_for_status()
        data = r.json()

    return data["message"]["content"]

# Your first local AI call!
response = await ask_gemma("Explain the data-to-insight pipeline in 3 sentences for a beginner.")
print(response)


### Step 3: Streaming output (token by token)

Streaming shows the response being generated in real time — this is how the Gemma chat in the sidebar works.


In [ ]:
import json as _json

async def ask_gemma_streaming(prompt: str, model: str = "gemma4:e2b") -> None:
    """Stream tokens from Ollama.
    Note: JupyterLite buffers the full response (no mid-stream output).
    Docker JupyterLab prints token-by-token.
    """
    try:
        from pyodide.http import pyfetch          # JupyterLite — buffered (no real streaming)
        print("⏳ Generating… (JupyterLite buffers the full response)")
        resp = await pyfetch(
            "http://localhost:11434/api/chat",
            method="POST",
            headers={"Content-Type": "application/json"},
            body=_json.dumps({
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "stream": False,
            }),
        )
        data = await resp.json()
        print(data["message"]["content"])
    except ImportError:
        import requests                            # Docker / server Jupyter — true streaming
        r = requests.post(
            "http://localhost:11434/api/chat",
            json={"model": model, "messages": [{"role": "user", "content": prompt}], "stream": True},
            stream=True, timeout=120,
        )
        r.raise_for_status()
        for line in r.iter_lines():
            if line:
                chunk = _json.loads(line)
                print(chunk.get("message", {}).get("content", ""), end="", flush=True)
        print()

await ask_gemma_streaming("What is cross-validation? One paragraph.")


### Step 4: Log your first AI interaction

Open (or create) `AI_USE.md` in your project root and add an entry using the disclosure template from M1 prompts.md.
